In [13]:
!pip install --upgrade transformers accelerate safetensors -q

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,   # better for Colab
    device_map="auto"
)

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [2]:
def generate_response(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response


In [4]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

messages = [
    {"role": "system", "content": "You are a helpful and friendly AI assistant."}
]

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day.")
        break

    messages.append({"role": "user", "content": user_input})

    response = generate_response(messages)

    print("Chatbot:", response)

    messages.append({"role": "assistant", "content": response})

Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.
You: hello
Chatbot: Hello! How can I assist you today?
You: What is Artificial Intelligence?
Chatbot: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn, and adapt like humans. It involves developing intelligent systems that can perform tasks such as image recognition, speech recognition, decision-making, problem-solving, and natural language processing.

The field of AI encompasses a wide range of technologies, including machine learning, deep learning, neural networks, robotics, autonomous
You: Who created Python?
Chatbot: Python was created byGuido van Rossum, an Italian computer scientist, in 1991. He began working on Python in 1986 while at university, but it wasn't until he left university to start work for a company called Sun Microsystems in 1990 that he started to develop the language. Python quickly became popular among developers, an